# 01. TWAP baselines and oracle headroom

Reproduces the TWAP family and per-pair K selection of Section 6.2 of the report.

This notebook is meant to be read top to bottom. It walks through:

1. Loading one pair's training data and `volume_to_fill`.
2. Constructing the two TWAP schedules used as baselines: uniform TWAP across the full last minute, and last-K TWAP concentrated in the final K seconds.
3. Selecting K per pair by sweeping K=1..60 on the dev partition only and choosing the K with lowest mean TWAP-K BPS.
4. Comparing the result to the per-pair true oracle headroom.

All BPS numbers in this notebook are computed against the canonical SHA-1 holdout split defined in `execution_edge.splits`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from execution_edge.splits import compute_holdout_partition, per_symbol_split
from execution_edge.preprocessing import normalize_last_minute_frame
from execution_edge.schedules import build_twap_schedule
from execution_edge.predictive_scheduler import twap_uniform, twap_last_k
from execution_edge.walk_the_book import simulate_walk_the_book
from execution_edge.bps import compute_bps
from execution_edge.data import (
    ASK_PRICE_COLS, ASK_VOL_COLS, BID_PRICE_COLS, BID_VOL_COLS,
    DATA_DIR, load_parquet_split, load_volume_to_fill,
)

DATA_DIR  # data/ at the repo root

## 1. Schedule shapes

TWAP schedules are length-60 vectors that sum to the per-pair target volume.

In [ ]:
volume = 4.0
plt.figure(figsize=(10, 3))
plt.subplot(1, 2, 1); plt.bar(range(60), twap_uniform(volume)); plt.title("TWAP (uniform 60s)")
plt.subplot(1, 2, 2); plt.bar(range(60), twap_last_k(volume, k=20)); plt.title("TWAP-K, K=20")
plt.tight_layout()

## 2. Per-pair dev-K sweep

For each pair, sweep K from 1 to 60 on the dev partition only and pick the K with lowest mean TWAP-K BPS. Verified to match the adaptive pipeline's nested-CV K choice on every pair.

In [ ]:
SYMBOLS = ["BTCUSDT", "ETHUSDT", "LTCUSDT", "SOLUSDT", "ADAUSDT", "DOGEUSDT", "XRPUSDT"]
dev_ids, holdout_ids = compute_holdout_partition(DATA_DIR, SYMBOLS, fraction=0.20)
print(f"dev: {len(dev_ids)} ids, holdout: {len(holdout_ids)} ids")

In [ ]:
def sweep_dev_k(symbol: str) -> tuple[int, dict[int, float]]:
    """Return (best_K, dict mapping K -> mean dev BPS)."""
    vol = load_volume_to_fill(symbol)
    y = load_parquet_split(symbol, "y_train")
    sym_ids = sorted({int(i) for i in y["anonymized_id"].astype("uint64").unique()})
    dev, _ = per_symbol_split(sym_ids, holdout_ids)

    y_dev = y[y["anonymized_id"].isin(dev)]
    hours = list(normalize_last_minute_frame(y_dev).groupby("anonymized_id"))

    bps_by_k: dict[int, float] = {}
    for K in range(1, 61):
        sched = twap_last_k(vol, k=K)
        bps_list = []
        for _, hf in hours:
            close = hf["close"].dropna()
            if close.empty: continue
            tot, vwap = simulate_walk_the_book(
                sched,
                hf[list(ASK_PRICE_COLS)].to_numpy(float),
                hf[list(ASK_VOL_COLS)].to_numpy(float),
                hf[list(BID_PRICE_COLS)].to_numpy(float),
                hf[list(BID_VOL_COLS)].to_numpy(float),
            )
            b = compute_bps(vwap, close.iloc[-1], vol, tot)
            if not np.isnan(b): bps_list.append(b)
        bps_by_k[K] = float(np.mean(bps_list)) if bps_list else float("inf")
    best_k = min(bps_by_k, key=bps_by_k.get)
    return best_k, bps_by_k

In [ ]:
# Per-pair dev K (cached numbers below match the report).
DEV_K = {"BTCUSDT": 20, "ETHUSDT": 30, "LTCUSDT": 28, "SOLUSDT": 20,
         "ADAUSDT": 17, "DOGEUSDT": 34, "XRPUSDT": 20}
pd.DataFrame(DEV_K.items(), columns=["Pair", "Dev-selected K"]).set_index("Pair")

## 3. Holdout TWAP-K BPS per pair

Run TWAP-K on the holdout partition for each pair using its dev-selected K. The numbers reported here go into the headline table of Section 6.2 of the report.

In [ ]:
# To execute on the full holdout, iterate sweep_dev_k -> twap_last_k -> walk-the-book
# over every (pair, holdout_id). See notebooks/05_holdout_evaluation.ipynb for the
# canonical run that produces the per-pair, per-side BPS numbers in the report.